# Challenge 2 — Détection de fraude explicable (IEEE-CIS)

**Objectif.** Détecter les transactions frauduleuses et **expliquer chaque décision** (SHAP) pour qu'un analyste puisse la vérifier.

**Ce notebook présente le projet de bout en bout**, avec une visualisation à chaque étape :

| Phase | Contenu |
|---|---|
| 1. Compréhension des données | sources, jointure, types, mémoire |
| 2. Qualité des données | valeurs manquantes, anomalies, colonnes quasi constantes, montants |
| 3. Exploration | déséquilibre, temps, montant, produit, appareil, corrélations |
| 4. Préparation | variables dérivées, découpage temporel, encodage, récapitulatif des décisions |
| 5. Modélisation | LightGBM avec arrêt précoce |
| 6. Évaluation | PR-AUC, matrice de confusion, seuils, coût métier, stabilité dans le temps |
| 7. Explicabilité | SHAP global, dépendance, explications locales |
| 8. Export | artefacts pour l'application Streamlit |

**Données.** Kaggle *IEEE-CIS Fraud Detection* : `train_transaction.csv` + `train_identity.csv` dans `data/`.

In [ ]:
%pip install -q lightgbm shap pyarrow

In [ ]:
import re, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    found = []
    for pat in ["fraud-xai*/src/pipeline.py", "*/fraud-xai*/src/pipeline.py", "*/*/fraud-xai*/src/pipeline.py", "*/*/*/fraud-xai*/src/pipeline.py"]:
        found = sorted(Path("/content/drive/MyDrive").glob(pat))
        if found: break
    if not found:
        raise FileNotFoundError("Projet introuvable sur le Drive : placez le dossier fraud-xai (contenant src/, data/, notebooks/) dans Mon Drive.")
    ROOT = found[0].parents[1]
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Projet :", ROOT)
sys.path.insert(0, str(ROOT / "src"))
import importlib
sys.path_importer_cache.pop(str(ROOT / "src"), None)   # évite un cache d'import périmé après un premier échec
importlib.invalidate_caches()

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display
import pipeline as P

FIG = ROOT / "artifacts" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED, ORANGE, GREEN = "#4C78A8", "#E45756", "#F58518", "#54A24B"
def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
steps = ["Sources\n(2 CSV)", "Jointure\net types", "Qualité\ndes données", "Exploration\ncorrélations", "Variables\net encodage",
         "Découpage\ntemporel", "LightGBM", "Évaluation\net seuils", "SHAP global\net local", "Application\nStreamlit"]
colors = [BLUE] * 2 + [ORANGE] * 2 + [GREEN] * 2 + ["#9467bd"] * 2 + [RED] * 2
fig, ax = plt.subplots(figsize=(15, 1.8)); ax.axis("off"); ax.set_xlim(-0.6, len(steps) - 0.4); ax.set_ylim(-0.6, 0.6)
for i, (s, c) in enumerate(zip(steps, colors)):
    ax.text(i, 0, s, ha="center", va="center", fontsize=8.5, bbox=dict(boxstyle="round,pad=0.45", fc=c, alpha=.35, ec="#555"))
    if i < len(steps) - 1:
        ax.annotate("", xy=(i + 0.66, 0), xytext=(i + 0.5, 0), arrowprops=dict(arrowstyle="->", color="#555"))
save("00_pipeline")

## 1. Compréhension des données


In [ ]:
p_tr, p_id = ROOT / "data" / "train_transaction.csv", ROOT / "data" / "train_identity.csv"
tx = pd.read_csv(p_tr, usecols=["TransactionID", "isFraud"])
idn = pd.read_csv(p_id, usecols=["TransactionID"])
n_cols_tx, n_cols_id = pd.read_csv(p_tr, nrows=0).shape[1], pd.read_csv(p_id, nrows=0).shape[1]
has_identity = tx.TransactionID.isin(idn.TransactionID)

display(pd.DataFrame({"lignes": [len(tx), len(idn)], "colonnes": [n_cols_tx, n_cols_id]},
                     index=["train_transaction", "train_identity"]))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
cov = has_identity.value_counts().reindex([True, False])
ax[0].bar(["avec identité", "sans identité"], cov.values, color=[BLUE, "#bbb"])
ax[0].set_title(f"Couverture de l'identité ({has_identity.mean():.0%} des transactions)")
fr = tx.groupby(has_identity).isFraud.mean().reindex([True, False]) * 100
ax[1].bar(["avec identité", "sans identité"], fr.values, color=[RED, "#bbb"])
ax[1].set_title("Taux de fraude selon la présence d'identité (%)")
save("01_jointure")
del tx, idn

In [ ]:
df = P.load_data(ROOT / "data")
print(df.shape)
df[["TransactionID", "TransactionDT", "TransactionAmt", "ProductCD", "card4", "card6", "DeviceType", "isFraud"]].head()

In [ ]:
is_text = lambda t: t == object or pd.api.types.is_string_dtype(t)
kinds = df.dtypes.map(lambda t: "texte (catégorielle)" if is_text(t) else str(t)).value_counts()
mem_now = df.memory_usage(deep=True).sum() / 1e9
n_light = sum(str(df[c].dtype) in ("float32", "int32") for c in df.columns)
mem_64 = mem_now + n_light * len(df) * 4 / 1e9

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].barh(kinds.index[::-1], kinds.values[::-1], color=BLUE); ax[0].set_title("Types des colonnes")
for i, v in enumerate(kinds.values[::-1]): ax[0].text(v, i, f" {v}", va="center")
ax[1].bar(["types 64 bits\n(par défaut)", "types allégés\n(utilisés)"], [mem_64, mem_now], color=["#bbb", GREEN])
ax[1].set_title("Mémoire du tableau (Go)")
for i, v in enumerate([mem_64, mem_now]): ax[1].text(i, v, f"{v:.1f}", ha="center", va="bottom")
save("02_types_memoire")

## 2. Qualité des données

In [ ]:
def family(c):
    for pat, name in [(r"V\d+", "V (Vesta)"), (r"C\d+", "C (comptages)"), (r"D\d+", "D (délais)"), (r"M\d", "M (correspondances)"),
                      (r"id_\d+", "id (identité)"), (r"card\d", "card"), (r"addr\d", "addr"), (r"dist\d", "dist")]:
        if re.fullmatch(pat, c): return name
    return "autres"

cols = df.columns.drop("isFraud")
fam = pd.Series({c: family(c) for c in cols})
miss = df.isna().mean()
tab = (pd.DataFrame({"famille": fam, "manquant": miss[cols] * 100}).groupby("famille")
       .agg(colonnes=("manquant", "size"), manquant_moyen=("manquant", "mean")).sort_values("manquant_moyen"))
display(tab.round(1))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].barh(tab.index, tab.manquant_moyen, color=ORANGE); ax[0].set_title("Valeurs manquantes moyennes par famille (%)")
ax[1].hist(miss[cols] * 100, bins=40, color=BLUE); ax[1].set_title("Répartition du taux de manquants par colonne")
ax[1].set_xlabel("% de valeurs manquantes"); ax[1].set_ylabel("Nombre de colonnes")
save("03_manquants_familles")

In [ ]:
samp = df.drop(columns="isFraud").sample(400, random_state=0).sort_index()
order = samp.isna().mean().sort_values().index
plt.figure(figsize=(11, 4))
plt.imshow(samp[order].isna().to_numpy(), aspect="auto", cmap="Greys", interpolation="nearest")
plt.xlabel("Colonnes (triées par taux de manquants croissant)"); plt.ylabel("400 transactions (ordre chronologique)")
plt.title("Carte des valeurs manquantes (noir = manquant)")
save("04_carte_manquants")

In [ ]:
S = df.sample(200_000, random_state=0) if len(df) > 200_000 else df
cand = [c for c in cols if 0.1 < miss[c] < 0.9]
rows = []
for c in cand:
    g = S.groupby(S[c].isna()).isFraud.mean() * 100
    if len(g) == 2: rows.append((c, g[True], g[False]))
mi = pd.DataFrame(rows, columns=["col", "si_manquant", "si_present"]).set_index("col")
mi["ecart"] = (mi.si_manquant - mi.si_present).abs()
top = mi.sort_values("ecart", ascending=False).head(12)[["si_manquant", "si_present"]].iloc[::-1]
top.plot.barh(figsize=(8, 5), color=[RED, BLUE])
plt.title("Taux de fraude (%) : valeur manquante vs présente"); plt.legend(["si manquant", "si présent"])
save("05_manquant_informatif")

In [ ]:
print("TransactionID unique       :", df.TransactionID.is_unique)
print("Ordre chronologique        :", df.TransactionDT.is_monotonic_increasing)
print("Montants <= 0              :", int((df.TransactionAmt <= 0).sum()))
print("Doublons sur (ID)          :", int(df.TransactionID.duplicated().sum()))

quasi = {}
for c in S.columns:
    vc = S[c].value_counts(normalize=True, dropna=False)
    if vc.iloc[0] > 0.99: quasi[c] = float(vc.iloc[0])
print(f"\nColonnes dont une valeur représente > 99 % des lignes : {len(quasi)}")
print(", ".join(list(quasi)[:15]) + (" …" if len(quasi) > 15 else ""))

In [ ]:
print(df.TransactionAmt.describe(percentiles=[.5, .9, .99, .999]).round(2).to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
cap = df.TransactionAmt.quantile(.995)
ax[0].hist(df.TransactionAmt.clip(upper=cap), bins=60, color=BLUE); ax[0].set_title("Montant brut (tronqué au 99,5e centile pour l'affichage)")
ax[1].hist(np.log1p(df.TransactionAmt), bins=60, color=GREEN); ax[1].set_title("Montant après log1p")
save("06_montants")

## 3. Exploration


In [ ]:
vc = df.isFraud.value_counts().sort_index()
print(f"Taux de fraude : {df.isFraud.mean():.2%}  ({vc[1]:,} fraudes / {len(df):,})")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(["Légitime", "Fraude"], vc.values, color=[BLUE, RED]); ax[0].set_yscale("log"); ax[0].set_title("Déséquilibre des classes (échelle log)")
by_day = df.assign(day=df.TransactionDT // 86400).groupby("day").isFraud.mean() * 100
ax[1].plot(by_day.index, by_day.values, lw=1); ax[1].set_title("Taux de fraude par jour (%)"); ax[1].set_xlabel("Jour depuis l'origine")
save("07_classes_temps")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for y_, c, l in [(0, BLUE, "Légitime"), (1, RED, "Fraude")]:
    ax[0].hist(np.log1p(df.loc[df.isFraud == y_, "TransactionAmt"]), bins=60, alpha=.6, color=c, density=True, label=l)
ax[0].set_title("Montant (log1p), densité"); ax[0].legend()
rate = df.groupby("ProductCD").isFraud.mean() * 100
ax[1].bar(rate.index, rate.values, color=ORANGE); ax[1].set_title("Taux de fraude par ProductCD (%)")
save("08_montant_produit")

In [ ]:
h = df.assign(hour=(df.TransactionDT // 3600) % 24).groupby("hour").isFraud.mean() * 100
d = df.groupby(df.DeviceType.fillna("inconnu")).isFraud.mean() * 100
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.index, h.values, marker="o"); ax[0].set_title("Taux de fraude par heure (%)")
ax[1].bar(d.index, d.values, color=GREEN); ax[1].set_title("Taux de fraude par type d'appareil (%)")
save("09_heure_appareil")

In [ ]:
cd = [c for c in df.columns if re.fullmatch(r"[CD]\d+", c)]
corr = S[cd].corr()
plt.figure(figsize=(8.5, 7))
im = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(cd)), cd, rotation=90, fontsize=7); plt.yticks(range(len(cd)), cd, fontsize=7)
plt.colorbar(im, fraction=.046); plt.title("Corrélation entre variables C* et D*")
save("10_correlation_CD")

In [ ]:
num = [c for c in S.select_dtypes("number").columns if c not in ("isFraud", "TransactionID")]
cor = S[num].corrwith(S.isFraud).dropna()
best = cor.reindex(cor.abs().sort_values(ascending=False).head(20).index).sort_values()
best.plot.barh(figsize=(7, 6), color=[RED if v > 0 else BLUE for v in best])
plt.title("Corrélation linéaire avec la fraude — top 20"); plt.xlabel("Corrélation de Pearson")
save("11_correlation_cible")

## 4. Préparation des données

In [ ]:
n_before = df.shape[1]
df = P.add_features(df)
new_cols = list(df.columns[n_before:])
print(f"{n_before} colonnes avant → {df.shape[1]} après (+{len(new_cols)}) : {new_cols}")

show = ["hour", "amt_log", "amt_vs_card_mean", "card_dt_since_last"]
fig, axs = plt.subplots(2, 2, figsize=(11, 6))
for ax, c in zip(axs.ravel(), show):
    v = df[c]; lo, hi = v.quantile([.01, .99])
    for y_, col, l in [(0, BLUE, "Légitime"), (1, RED, "Fraude")]:
        ax.hist(v[df.isFraud == y_].clip(lo, hi).dropna(), bins=50, alpha=.6, density=True, color=col, label=l)
    ax.set_title(c)
axs[0, 0].legend()
save("12_variables_derivees")

In [ ]:
train, val, test = P.temporal_split(df)
parts = [("train", train, BLUE), ("val", val, ORANGE), ("test", test, RED)]
for n, d_, _ in parts:
    print(f"{n:5s} {len(d_):>7,} lignes | jours {d_.TransactionDT.min()//86400:>3.0f}→{d_.TransactionDT.max()//86400:>3.0f} | fraude {d_.isFraud.mean():.2%}")

daily = df.groupby(df.TransactionDT // 86400).size()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [2.2, 1]})
ax[0].plot(daily.index, daily.values, color="#555", lw=1)
for n, d_, c in parts:
    ax[0].axvspan(d_.TransactionDT.min() // 86400, d_.TransactionDT.max() // 86400, color=c, alpha=.25, label=f"{n} ({len(d_):,})")
ax[0].legend(); ax[0].set_title("Découpage chronologique"); ax[0].set_xlabel("Jour"); ax[0].set_ylabel("Transactions / jour")
ax[1].bar([n for n, _, _ in parts], [d_.isFraud.mean() * 100 for _, d_, _ in parts], color=[c for _, _, c in parts])
ax[1].set_title("Taux de fraude par partie (%)")
save("13_decoupage_temporel")

In [ ]:
feats, enc, (Xtr, Xva, Xte) = P.prepare(train, val, test)
cat_cols = list(enc)
print(f"{len(feats)} variables dont {len(cat_cols)} catégorielles | X_train {Xtr.shape} | types : {dict(Xtr.dtypes.astype(str).value_counts())}")

card = pd.DataFrame({"avant": [train[c].nunique() for c in cat_cols], "après": [len(enc[c]) + int(train[c].nunique() > len(enc[c])) for c in cat_cols]}, index=cat_cols)
card = card.sort_values("avant", ascending=False).head(12).iloc[::-1]
card.plot.barh(figsize=(8, 5), color=["#bbb", GREEN], logx=True)
plt.title("Modalités avant / après regroupement des rares (« autre » inclus si utilisé, échelle log)")
save("14_encodage_cardinalite")

for c in [c for c in ["ProductCD", "card4"] if c in enc]:
    print(f"\nCodage de {c} :")
    display(pd.DataFrame({"modalité": enc[c] + ["(autre)"], "code": range(len(enc[c]) + 1)}).set_index("code").T)

In [ ]:
n_raw = n_cols_tx + n_cols_id - 1          # colonnes brutes après jointure (TransactionID compté une fois)
recap = pd.DataFrame([
    ("Colonnes brutes après jointure", n_raw),
    ("+ variables dérivées", len(new_cols)),
    ("− identifiant, cible, horodatage (non utilisés comme variables)", -3),
    ("= variables fournies au modèle", len(feats)),
    ("dont catégorielles encodées", len(cat_cols)),
], columns=["étape", "colonnes"]).set_index("étape")
display(recap)

## 5. Modélisation LightGBM


In [ ]:
pd.Series(P.PARAMS).to_frame("hyperparamètre")

In [ ]:
model = P.train_model(Xtr, train.isFraud, Xva, val.isFraud, cat_cols)
print("Meilleure itération :", model.best_iteration_)
ev = model.evals_result_["valid_0"]["auc"]
plt.plot(ev); plt.axvline(model.best_iteration_, c="r", ls="--", label=f"arrêt à {model.best_iteration_}")
plt.xlabel("Itération"); plt.ylabel("AUC (validation)"); plt.title("Courbe d'apprentissage"); plt.legend()
save("15_courbe_apprentissage")

## 6. Évaluation sur le futur 

In [ ]:
pva = model.predict_proba(Xva)[:, 1]; pte = model.predict_proba(Xte)[:, 1]
yva, y = val.isFraud.to_numpy(), test.isFraud.to_numpy()
amt = test.TransactionAmt.to_numpy()
thr_f1 = P.best_f1_threshold(yva, pva)
thr_cost = P.cost_threshold(yva, pva, val.TransactionAmt.to_numpy())
metrics = P.evaluate(y, pte, amt, thr_f1, thr_cost)
pd.Series(metrics).to_frame("valeur")

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
fpr, tpr, _ = roc_curve(y, pte); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", c="grey")
ax[0].set_title(f"ROC (AUC = {metrics['roc_auc']:.3f})")
pr, rc, _ = precision_recall_curve(y, pte); ax[1].plot(rc, pr); ax[1].axhline(y.mean(), ls="--", c="grey")
ax[1].set_title(f"Précision-rappel (AP = {metrics['pr_auc']:.3f})"); ax[1].set_xlabel("Rappel"); ax[1].set_ylabel("Précision")
ConfusionMatrixDisplay(np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]),
                       display_labels=["Légitime", "Fraude"]).plot(ax=ax[2], colorbar=False, values_format="d")
ax[2].set_title(f"Confusion (seuil {thr_f1:.2f})")
save("16_performance")

In [ ]:
grid = np.linspace(0.01, 0.99, 99)
costs = [P.total_cost(y, pte, amt, t) for t in grid]
plt.plot(grid, costs)
plt.axvline(thr_cost, c="r", ls="--", label=f"seuil coût = {thr_cost:.2f}")
plt.axvline(thr_f1, c="g", ls="--", label=f"seuil F1 = {thr_f1:.2f}")
plt.xlabel("Seuil"); plt.ylabel("Coût total ($)"); plt.legend(); plt.title("Coût métier selon le seuil (test)")
save("17_cout_seuil")
print(f"Pertes sans modèle : {metrics['cost_no_model']:,.0f} $ | avec modèle : {metrics['cost_with_model']:,.0f} $ ({metrics['cost_reduction']:.0%} évités)")

In [ ]:
from sklearn.metrics import roc_auc_score
wk = ((test.TransactionDT - test.TransactionDT.min()) // (7 * 86400)).to_numpy()
res = {int(w): roc_auc_score(y[wk == w], pte[wk == w]) for w in np.unique(wk) if len(np.unique(y[wk == w])) == 2}
plt.bar(list(res), list(res.values()), color=BLUE); plt.ylim(min(res.values()) - .05, 1)
plt.title("AUC par semaine du jeu de test (stabilité dans le temps)"); plt.xlabel("Semaine"); save("18_auc_semaine")

## 7. Explicabilité globale 

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(Xte), min(3000, len(Xte)), replace=False)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xte.iloc[idx]); sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, Xte.iloc[idx], max_display=20, show=False); save("19_shap_beeswarm")
shap.summary_plot(sv, Xte.iloc[idx], plot_type="bar", max_display=20, show=False); save("20_shap_bar")

In [ ]:
top = np.abs(sv).mean(0).argsort()[::-1][:3]
for j in top:
    shap.dependence_plot(feats[j], sv, Xte.iloc[idx], interaction_index=None, show=False); save(f"21_dependance_{feats[j]}")

## 8. Explicabilité locale : trois cas à discuter
Une fraude détectée, une fausse alerte, une fraude manquée.


In [ ]:
base = float(np.ravel(explainer.expected_value)[-1])
cases = {
    "Fraude détectée": np.where((y == 1) & (pte >= thr_f1))[0],
    "Faux positif":    np.where((y == 0) & (pte >= thr_f1))[0],
    "Fraude manquée":  np.where((y == 1) & (pte < thr_f1))[0],
}
for k, (name, ids) in enumerate(cases.items()):
    if len(ids) == 0: continue
    i = ids[np.argmin(pte[ids])] if name == "Fraude manquée" else ids[np.argmax(pte[ids])]
    print(f"{name} → ligne {i} | score {pte[i]:.3f} | montant {amt[i]:.2f} $")
    s = explainer.shap_values(Xte.iloc[[i]]); s = s[1] if isinstance(s, list) else s
    shap.plots.waterfall(shap.Explanation(values=s[0], base_values=base, data=Xte.iloc[i].values, feature_names=feats), max_display=12, show=False)
    plt.title(name); save(f"22_waterfall_{k}")

In [ ]:
art = P.export_artifacts(model, enc, feats, Xte, test, metrics)
print("Artefacts écrits dans", art)
print("Lancer l'application :  streamlit run streamlit_app.py")